In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextDataset, Trainer, TrainingArguments
from transformers import pipeline

# 1. Recopilar frases de Rajoy en un archivo
# 2. Cargar GPT-2 preentrenado
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. Fine-tuning con tus datos
# ... (entrenar con las frases)

# 4. Generar
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)
generator("La cuestión es que", max_length=50)

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 no tiene pad_token por defecto
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# 1) Archivo de texto con frases (una o varias por línea)
#    ejemplo: rajoy.txt
dataset = load_dataset("text", data_files={"train": "rajoy.txt", "validation": "rajoy_val.txt"})

# 2) Tokenizar
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

# 3) Data collator para Causal LM (mlm=False)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 4) Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./gpt2-rajoy",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,  # quítalo si no tienes GPU compatible
    report_to="none"
)

# 5) Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

trainer.train()

# 6) Guardar
trainer.save_model("./gpt2-rajoy")
tokenizer.save_pretrained("./gpt2-rajoy")

In [ ]:

!pip -q install transformers datasets peft accelerate


KeyboardInterrupt: 

In [ ]:

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model.config.pad_token_id = tokenizer.eos_token_id

# LoRA config (GPT-2 usa módulos tipo c_attn/c_proj)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn", "c_proj"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()  # verás que entrena solo una parte pequeña

dataset = load_dataset("text", data_files={"train": "rajoy.txt", "validation": "rajoy_val.txt"})

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = dataset.map(tok, batched=True, remove_columns=["text"])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="./gpt2-rajoy-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,   # en LoRA suele ir algo más alto
    fp16=True,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator
)

trainer.train()
model.save_pretrained("./gpt2-rajoy-lora")
tokenizer.save_pretrained("./gpt2-rajoy-lora")